# Cum am construit `app/app.py` — tutorial Gradio (pas cu pas)

Ideea Gradio, într-o propoziție: **o funcție Python devine o interfață web.**
Tu scrii funcția, Gradio face caseta, butonul și layout-ul.

Construim app-ul incremental, exact în ordinea în care e scris `app.py`:
funcția simplă -> mai multe input-uri -> tab Chat -> starea partajată ->
regula subiect/știre -> tab Agent -> punem tab-urile împreună -> recapitulare.

Inspirat din [Gradio Quickstart](https://www.gradio.app/guides/quickstart).
Regula tutorialului: **cât mai simplu, doar esențialul.**

> `app/app.py` este doar un strat subțire de Gradio peste `core/` (agent, graph),
> construit în cursurile C2–C7. Aici nu rescriem `core/` — îl chemăm.
> Ca să ruleze fără chei API, folosim un backend fals.

In [1]:
%pip install -q gradio
import gradio as gr
print("Gradio", gr.__version__)


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio 6.14.0


## 1. Cel mai simplu Gradio

`gr.Interface` are nevoie de 3 lucruri: `fn` (funcția), `inputs`, `outputs`.

In [2]:
def saluta(nume):
    return "Salut, " + nume

gr.Interface(fn=saluta, inputs="text", outputs="text").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Atât. Caseta, butonul *Submit*, totul l-a făcut Gradio. Pe asta se construiește
toată aplicația.

## 2. Mai multe input-uri = o listă

Tab-urile noastre au mai multe câmpuri. Dacă funcția are mai multe argumente,
dai o **listă** la `inputs` (ordinea = ordinea argumentelor).

In [3]:
def combina(text, optiune, numar):
    return f"[{optiune} @ {numar}] {text}"

gr.Interface(
    fn=combina,
    inputs=[gr.Textbox(label="Text"),
            gr.Dropdown(["a", "b"], value="a", label="Opțiune"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Număr")],
    outputs=gr.Textbox(label="Rezultat"),
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Reține tiparul `[text, dropdown, slider] -> funcție -> text`. **Asta e un tab.**

## 3. Backend fals (ca să rulăm fără chei API)

În `app.py` real, sus, sunt 2 importuri din `core/` (construite în C6–C7):

```python
from core.agent import generate_agent_response   # un agent RAG
from core.graph import run_thread                 # dezbatere multi-agent
```

Aici le înlocuim cu funcții-jucărie. Restul codului rămâne identic ca structură.

In [4]:
def fake_llm(prompt):
    return "(răspuns simulat) despre: " + prompt[:70]

def fake_agent(slug, stimulus):
    voci = {"anti_sistem": "Instituțiile par din nou rupte de oameni.",
            "pro_european": "Să discutăm pe baza procedurilor."}
    return voci.get(slug, f"[{slug}] {stimulus[:50]}")

AGENTS = [("Anti-sistem", "anti_sistem"), ("Pro-european", "pro_european")]
print("backend fals pregătit")

backend fals pregătit


## 4. Primul tab real: Chat

În `app.py`, tab-ul Chat e funcția `chat()` + un `gr.Interface`. O reproducem
cu `fake_llm`.

In [5]:
def chat(prompt):
    return fake_llm(prompt) if prompt.strip() else "Scrie un prompt."

gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Întrebare / prompt", lines=4),
    outputs=gr.Textbox(label="Răspuns", lines=10),
    title="Chat",
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tab-ul Chat complet, fără să scriem vreun buton. Gradio l-a făcut.

## 5. Starea partajată: `CFG` și `ART`

Tab-ul **Setări** alege modelul și (opțional) încarcă o știre. Celelalte tab-uri
trebuie să **vadă** acea știre. Soluția minimă: două dicționare la nivel de modul.

- `CFG` = provider / model / temperatură
- `ART` = textul + titlul știrii încărcate

Setări **scrie** în ele, restul tab-urilor **citesc**. (Alternativa „canonică"
ar fi `gr.State`; varianta minimă alege simplitatea.)

Truc din `app.py`: provider + model sunt **un singur dropdown**
(`"provider|model"`) — imposibil să fie nepotrivite.

In [6]:
CFG = {"provider": "gemini", "model": "gemini-2.5-flash-lite", "temp": 0.3}
ART = {"text": "", "title": ""}

MODEL_CHOICES = [("gemini · gemini-2.5-flash-lite", "gemini|gemini-2.5-flash-lite"),
                 ("deepseek · deepseek-chat",       "deepseek|deepseek-chat")]

def setup(model_choice, temperature, fake_url):
    provider, model = model_choice.split("|", 1)     # despărțim "provider|model"
    CFG.update(provider=provider, model=model, temp=temperature)
    if fake_url.strip():
        ART.update(text=f"Text fals al știrii de la {fake_url}", title=fake_url)
        return f"Setări salvate. Știre ACTIVĂ: {fake_url}"
    ART.update(text="", title="")
    return f"Setări salvate ({provider} · {model}). Fără știre."

gr.Interface(
    fn=setup,
    inputs=[gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1],
                        label="Provider · Model"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
            gr.Textbox(label="URL știre (gol = fără știre)")],
    outputs=gr.Textbox(label="Stare"),
    title="Setări",
).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 6. Regula cheie: subiectul intră *peste* știre

`_subject()` decide ce primește un agent:

- **știre + subiect** -> vorbim despre subiect, dar **în contextul știrii**
- **doar știre** -> vorbim despre știre
- **doar subiect** -> vorbim doar despre subiect

In [7]:
def _subject(typed):
    typed = (typed or "").strip()
    news = ART["text"].strip()
    if news and typed:
        return f"{typed}\n\n[În contextul acestei știri:]\n{news[:600]}"
    if news:
        return news[:700]
    return typed

ART.update(text="Știre despre UE și energie.")
print(_subject("Bolojan"))     # subiect peste știre
ART.update(text="")
print(_subject("Bolojan"))     # doar subiect

Bolojan

[În contextul acestei știri:]
Știre despre UE și energie.
Bolojan


## 7. Tab-ul Agent

Tab-ul Agent = `_subject()` + chemarea backend-ului. În `app.py` real,
`fake_agent` e `generate_agent_response` din `core.agent` (C6).

In [8]:
def agent(text, slug):
    s = _subject(text)
    if not s.strip():
        return "Încarcă o știre sau scrie un subiect."
    return fake_agent(slug, s)               # în app: generate_agent_response(...)

gr.Interface(
    fn=agent,
    inputs=[gr.Textbox(label="Subiect (intră peste știre, dacă e încărcată)",
                       lines=3),
            gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    outputs=gr.Textbox(label="Comentariu", lines=10),
    title="Agent",
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tab-urile **Rezumat**, **Toți agenții** și **Dezbatere** au exact același tipar:
o funcție + un `gr.Interface`. Doar funcția diferă (rezumat / loop pe roluri /
`core.graph.run_thread`).

## 8. Punem tab-urile împreună

`app.py` are 6 tab-uri cu o **temă comună**. `gr.TabbedInterface` nu acceptă
`theme=` pe toate versiunile, așa că facem ce face el intern: un `gr.Blocks`
cu temă, `gr.Tabs`, și randăm fiecare `Interface` cu `.render()`.

In [9]:
tab_setup = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label="Provider · Model"),
     gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
     gr.Textbox(label="URL știre")],
    gr.Textbox(label="Stare"), title="Setări")

tab_chat = gr.Interface(chat, gr.Textbox(label="Prompt", lines=3),
    gr.Textbox(label="Răspuns", lines=8), title="Chat")

tab_agent = gr.Interface(agent,
    [gr.Textbox(label="Subiect", lines=3),
     gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    gr.Textbox(label="Comentariu", lines=8), title="Agent")

TABS = [("Setări", tab_setup), ("Chat", tab_chat), ("Agent", tab_agent)]

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown("# EchoChamber Studio")
    with gr.Tabs():
        for nume, iface in TABS:
            with gr.Tab(nume):
                iface.render()

demo.launch()

/var/folders/rj/pj52rxcd7gddmhj55h5kcqnw0000gn/T/ipykernel_46464/3885253093.py:17: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Acesta e scheletul exact din `app.py`. În aplicația reală sunt 6 tab-uri în
loc de 3, iar funcțiile cheamă `core/` în loc de `fake_*`.

## 9. Recapitulare

**Ce face Gradio (tot tutorialul, pe scurt):**

1. `gr.Interface(fn, inputs, outputs)` — o funcție devine interfață.
2. Mai multe input-uri = o listă. Un astfel de bloc = un tab.
3. `CFG` / `ART` (dict-uri de modul) = starea partajată: Setări scrie, restul citesc.
4. `_subject()` = regula subiect-peste-știre.
5. `gr.Blocks` + `gr.Tabs` + `.render()` = cele 6 tab-uri cu temă comună.

**Dependențe (din structura repo):** `app/app.py` cheamă doar `core/` —
nu rescrie nimic.

| Tab(uri) | Funcție în app.py | Backend | Curs |
|---|---|---|---|
| Setări · Chat · Rezumat | `setup` · `chat` · `summary` | apel LLM direct | C2 |
| Agent | `agent` -> `_agent` | `core.agent` (FAISS + rol) | C5 + C6 |
| Toți agenții | `all_agents` | loop pe `roles.yaml` -> `core.agent` | C6 |
| Dezbatere | `debate` | `core.graph.run_thread` (LangGraph) | C7 |

`core.agent` -> `core.retriever` (FAISS) + `roles.yaml` + LLM.
`core.graph` orchestrează `core.agent` (round-robin). Singura punte offline->runtime:
vectorstore-urile construite offline, citite de retriever la fiecare cerere.

**Mesajul cheie:** aplicația nu e un proiect nou. E un strat subțire Gradio
peste funcțiile din C2–C7. Fiecare tab = un buton peste o funcție de curs.

## 10. Tema 3 — Extensia mea individuală

Adaug 3 modificări vizibile peste scheletul din capitolul 8:

1. **Tab nou „Despre & Etică"** — explică ce face aplicația, limitele simulării și disclaimer-ul AI.
2. **Funcție nouă `count_words_and_chars()`** — numără cuvintele și caracterele din răspunsul agentului, ca să verificăm dacă agentul respectă limita de propoziții cerută în rol.
3. **Element de design** — temă schimbată (`Monochrome` cu accent pe roșu portocaliu, exact culoarea bulei `anti_sistem`), titlu cu emoji, subtitlu cu data ultimei rulări, și un mesaj cu instrucțiuni vizibil sub titlu.

### Ce am adăugat
Un tab informativ + o utilitate de inspecție lingvistică + un refresh stilistic care leagă vizual aplicația de bula pe care am lucrat-o (anti-sistem).

### Funcție nouă creată
`count_words_and_chars(text)` — primește un text și returnează un raport formatat cu numărul de cuvinte, caractere și propoziții estimate. Folosit într-un tab separat „Inspecție răspuns" unde lipești output-ul unui agent și verifici dacă respectă regula „maxim 3 propoziții" din `role_02.yaml`.

### Schimbare design
Trecere de la `Soft(primary_hue="orange")` la `Monochrome(primary_hue="red")`, titlu cu emoji 🗣️, subtitlu cu data, și un blockquote cu disclaimer-ul AI vizibil pe toate tab-urile.

### Ce aș îmbunătăți mai departe
Aș conecta `count_words_and_chars` direct la output-ul tab-ului „Agent" (în loc de copy-paste manual), aș adăuga un buton „Curăță thread" în tab-ul „Dezbatere", și aș salva istoricul conversațiilor într-un fișier local pentru analiză ulterioară.

In [11]:
import gradio as gr
from datetime import datetime
import re

# === FUNCȚIE NOUĂ: inspecție lingvistică ===
def count_words_and_chars(text):
    """Numara cuvinte, caractere si propozitii dintr-un text."""
    if not text or not text.strip():
        return "Niciun text de analizat."
    
    text = text.strip()
    words = len(text.split())
    chars = len(text)
    chars_no_spaces = len(text.replace(" ", ""))
    # numar propozitii estimate (split pe . ! ?)
    sentences = len([s for s in re.split(r'[.!?]+', text) if s.strip()])
    
    raport = f"""📊 Raport de inspecție lingvistică

Cuvinte:              {words}
Caractere (cu spații): {chars}
Caractere (fără spații): {chars_no_spaces}
Propoziții estimate:  {sentences}

Respectă limita de 3 propoziții?  {'✅ DA' if sentences <= 3 else '❌ NU (depășește)'}
"""
    return raport


# === TAB NOU 1: Inspecție răspuns ===
tab_inspectie = gr.Interface(
    count_words_and_chars,
    gr.Textbox(label="Lipește răspunsul agentului aici", lines=6, placeholder="Copiază aici un comentariu generat de agent..."),
    gr.Textbox(label="Raport", lines=10),
    title="🔍 Inspecție răspuns"
)


# === TAB NOU 2: Despre & Etică ===
def info_aplicatie(_):
    return """
🗣️ **EchoChamber Studio** simulează agenți conversaționali bazați pe bule discursive 
extrase din comentarii YouTube românești.

**Ce face aplicația:**
- Fiecare agent are un rol (definit în `roles.yaml`) și un corpus propriu (50 comentarii din C5).
- Răspunsurile sunt generate de un LLM (Gemini sau DeepSeek) cu context recuperat din FAISS.
- Tab-ul Dezbatere folosește LangGraph pentru a orchestra mai mulți agenți într-un thread.

**Limite și etică:**
- Agenții NU sunt persoane reale. Sunt simulări discursive.
- Selecția corpusului introduce bias: păstrăm comentariile cele mai marcate stilistic, eliminăm vocile moderate.
- Output-ul nu trebuie folosit ca sondaj de opinie sau ca dovadă despre ce gândește un grup social.
- Toate răspunsurile sunt generate de AI — necesită moderare umană înainte de orice utilizare publică.

**Bula pe care am lucrat eu:** `anti_sistem` — cetățean român furios și dezamăgit de instituții, BOR, presă mainstream și justiție.
"""

tab_despre = gr.Interface(
    info_aplicatie,
    gr.Textbox(visible=False, value="trigger"),
    gr.Markdown(),
    title="ℹ️ Despre & Etică",
    submit_btn="Afișează informații"
)


# === TAB-URILE EXISTENTE (reutilizate) ===
# (presupunem ca setup, chat, agent, MODEL_CHOICES, AGENTS sunt definite mai sus)

tab_setup_v2 = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label="Provider · Model"),
     gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
     gr.Textbox(label="URL știre")],
    gr.Textbox(label="Stare"), title="⚙️ Setări")

tab_chat_v2 = gr.Interface(chat, gr.Textbox(label="Prompt", lines=3),
    gr.Textbox(label="Răspuns", lines=8), title="💬 Chat")

tab_agent_v2 = gr.Interface(agent,
    [gr.Textbox(label="Subiect", lines=3),
     gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    gr.Textbox(label="Comentariu", lines=8), title="🎭 Agent")


# === DESIGN SCHIMBAT: temă, titlu, subtitlu, disclaimer ===
TABS_V2 = [
    ("Setări", tab_setup_v2),
    ("Chat", tab_chat_v2),
    ("Agent", tab_agent_v2),
    ("Inspecție", tab_inspectie),    # tab nou
    ("Despre", tab_despre),           # tab nou
]

azi = datetime.now().strftime("%d.%m.%Y")

with gr.Blocks(theme=gr.themes.Monochrome(primary_hue="red")) as demo_v2:
    gr.Markdown(f"""
    # 🗣️ EchoChamber Studio — student_02 (anti_sistem)
    
    *Simulator de bule discursive românești · ultima rulare: {azi}*
    
    > ⚠️ Toate răspunsurile sunt generate de AI pe baza unui corpus selectat. Nu reprezintă opinii reale ale niciunei persoane sau grup.
    """)
    
    with gr.Tabs():
        for nume, iface in TABS_V2:
            with gr.Tab(nume):
                iface.render()

demo_v2.launch()

/var/folders/rj/pj52rxcd7gddmhj55h5kcqnw0000gn/T/ipykernel_46464/742268222.py:97: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome(primary_hue="red")) as demo_v2:


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
